In [ ]:
# Imports
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import animation
import salvus.namespace as sn
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers.wavefield_output import WavefieldOutput, wavefield_output_to_xarray
import obspy
from scipy import signal
from __future__ import annotations

import typing

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import scipy.signal

**Mesh Setup**
Three-layered mesh with snow, weak layer (currently same parameters sas snow) and air, from 0 to 400m with 3m height

In [ ]:
# Mesh generation

# Create Salvus Project domain
def setup_salvus_project(project_dir, x0=0, x1=400, y0=0, y1=3):
    """Initializes or loads a Salvus project."""
    domain = sn.domain.dim2.BoxDomain(x0=x0, x1=x1, y0=y0, y1=y1)
    p = sn.Project.from_domain(path=project_dir, domain=domain, load_if_exists=True)
    return p, domain

# Create Mesh
def build_layered_mesh(p, f0, ...):
    """
    Copy your mesh generation code here (the splines, layers, ABCs).
    """
   
# Layered model setup: three layers ordered as snow, slab, air (top to bottom).


x_min, x_max = 0.0, 400.0
# Geometry (y-coordinates: higher = higher depth, measured downward):
# Snow: from y=3.0 m (top) to y=2.25 m
# Slab: from y=2.25 m to y=1.5 m  
# Air:  from y=1.5 m to y=0.0 m (bottom)

slab_top = 3.0
slab_bottom = 2.25
wl_top = 2.25
wl_bottom = 1.5
air_top = 1.5
air_bottom = 0.0

# Boundaries from top to bottom -> 3 layers.
# Must be strictly ordered: top to bottom (decreasing y).
layers_x = [
    np.array([x_min, x_max]),  # snow top boundary
    np.array([x_min, x_max]),  # snow-slab interface
    np.array([x_min, x_max]),  # slab-air interface
    np.array([x_min, x_max]),  # air bottom boundary
]
layers_y = [
    np.array([slab_top, slab_top]),      # y = 3.0
    np.array([slab_bottom, slab_bottom]),  # y = 2.25
    np.array([wl_bottom, wl_bottom]),  # y = 1.5
    np.array([air_bottom, air_bottom]),    # y = 0.0
]

# Material parameters by region index [snow, slab, air].
vp = np.array([109.4, 109.4, 330.0]) 
vs = np.array([67.9366, 67.9366, 0.0])   
rho = np.array([250.0, 250.0, 1.225])

interpolation_styles = ["linear"] * len(layers_x)
splines = sn.toolbox.get_interpolating_splines(layers_x, layers_y, kind=interpolation_styles)

# Maximum frequency for meshing
# Phyiscical frequency of wavelet using fft
n_fft = len(ricker_base)
fft_vals = np.fft.rfft(ricker_base)
frequencies = np.fft.rfftfreq(n_fft, d=dt)
amplitude_spectrum = np.abs(fft_vals)

# Finding amplitude
cumulative_amplitude = np.cumsum(amplitude_spectrum)
total_amplitude = cumulative_amplitude[-1]

# Frequency where 95% of spactrum is contained
percentile_95_idx = np.where(cumulative_amplitude >= 0.95 * total_amplitude)[0][0]
max_frequency_source = frequencies[percentile_95_idx]
max_frequency = float(max_frequency_source)

print(f"Base Ricker Center Frequency (f0): {f0} Hz")
print(f"95th Percentile Source Frequency Content: {max_frequency:.1f} Hz")



# One per layer pair; last value keeps meshing stable above acoustic air.
slowest_velocities = np.array([67.0, 67.0,  67.0])

# calculating the wavelength of the center freqnecy (lambda = v / f)
wavelengths = slowest_velocities[0] / 20
print(f"Calculated minimum thickness for ABC: {wavelengths} m") # Absorbing boundary should be at least 3.5 wavelengths 


mesh, bnd = sn.toolbox.generate_mesh_from_splines_2d(
    x_min=x_min,
    x_max=x_max,
    splines=splines,
    elements_per_wavelength=2.5,
    maximum_frequency=max_frequency,
    use_refinements=True,
    slowest_velocities=slowest_velocities,
    # absorbing_boundaries=(["x0", "x1", "y0"], 2.8), # check this with the printed code below
)
mesh = np.sum(mesh)
mesh.attach_global_variable("max_dist_ABC", bnd)
mesh.attach_global_variable("ABC_side_sets", ", ".join(["x0", "x1", "y0"]))
mesh.attach_global_variable("ABC_vel", float(min(vs[vs > 0])))
mesh.attach_global_variable("ABC_freq", max_frequency / 2.0)
mesh.attach_global_variable("ABC_nwave", 5.0)

nodes = mesh.get_element_nodes()[:, :, 0]
vp_a, vs_a, ro_a = np.ones((3, *nodes.shape))
for _i, (vp_val, vs_val, ro_val) in enumerate(zip(vp, vs, rho)):
    idx = np.where(mesh.elemental_fields["region"] == _i)
    vp_a[idx] = vp_val
    vs_a[idx] = vs_val
    ro_a[idx] = ro_val

for k, v in zip(["VP", "VS", "RHO"], [vp_a, vs_a, ro_a]):
    mesh.attach_field(k, v)

mesh_3layer = sn.toolbox.detect_fluid(mesh)
print("Three-layer mesh built.")
print(f"  SLab: y = [{slab_top:.2f}, {slab_bottom:.2f}] m, vs={vs[0]:.0f} m/s")
print(f"  Weak layer: y = [{wl_top:.2f}, {wl_bottom:.2f}] m, vs={vs[1]:.0f} m/s")
print(f"  Air layer:  y = [{air_top:.2f}, {air_bottom:.2f}] m, vs={vs[2]:.0f} m/s")

# calculating how wide one element in the mesh is
# first disrtance between two nodes in the x direction
node_distances = np.sqrt(np.sum(np.diff(nodes, axis=0)**2, axis=1))
average_node_distance = np.mean(node_distances)
print(f"Average node distance in the mesh: {average_node_distance:.2f} m")

# there should be 3.5 times the wavelength thickness in the abc
abc_thickness = wavelengths * 3.5
print(f"Recommended ABC thickness based on wavelength: {abc_thickness:.2f} m")
# translating this to number of elements: abc_thickness / average_node_distance
abc_thickness_in_elements = abc_thickness / average_node_distance
print(f"Recommended ABC thickness in number of elements: {abc_thickness_in_elements:.1f} elements")


    print("Mesh successfully generated and added to project.")

**Source Setup** Sub-rayleigh or supershear with fixed distance (input)

In [ ]:
def generate_moving_sources(target_vprop, x_positions, y_src, f0=20.0, sampling_rate=10000.0):
    """
    Generates the Custom STF array with the Hanning tapers.
    """
    
    
        # Simulation constants
    f0 = 20.0
    sampling_rate = 10000.0
    dt = 1.0 / sampling_rate

    step =0.1
    x_positions = np.arange(30.0, 270.0, step)
    # target_vprop = 1.6*67.9366 # supershear: 1.6*vs, according to trottet et al, 2022
    target_vprop = 0.6*67.9366 # sub-rayleigh: 0.6*vs, according to trottet et al, 2022
    delay_between_sources = step / target_vprop
    print(f"Time step between sources: {delay_between_sources:.4f} s")

    y_src = 2.625

    # Shared time setup (half width can be the same for all the wavelets )
    half_width = 0.08
    pre_delay = half_width

    last_source_delay = (len(x_positions) - 1) * delay_between_sources
    t_max = pre_delay + last_source_delay + half_width + 0.5
    t_sim = np.arange(0, t_max, dt)
    print(f"t_sim length: {len(t_sim)} samples, duration: {t_sim[-1]:.3f} s")

    t_local = np.arange(-half_width, half_width, dt)
    half_samples = len(t_local) // 2

    # Define Bases of wavelets
    # Ricker 
    ricker_base = (
        (1.0 - 2.0 * (np.pi * f0 * t_local) ** 2)
        * np.exp(-((np.pi * f0 * t_local) ** 2))
    )

    # # Morlet
    # morlet_base = (
    #     np.exp(-((t_local / half_width) ** 2)) 
    #     * np.cos(2 * np.pi * f0 * t_local)
    # )

    # # Gaussian
    # gaussian_base = np.exp(-((t_local / half_width) ** 2))


    # Weigths setup
    base_mxx =  1.0
    base_myy = -1.0
    base_mxy =  1.0
    
    srcs = []
        N_taper = int(half_width / delay_between_sources) + 1# taper over exactly one wavelet-width worth of sources
    print(f"Tapering first {N_taper} sources")

    taper = np.ones(len(x_positions))
    taper[:N_taper] = np.hanning(2 * N_taper)[:N_taper] # rising half of Hanning window: for first source, it needs to be ramped up
    taper[-N_taper:] = np.hanning(2 * N_taper)[N_taper:] # for last source, needs to be ramped down 

    random_weight_xx = taper.copy()
    random_weight_yy = taper.copy()
    random_weight_xy = taper.copy()
    weight_array = np.array([random_weight_xx, random_weight_yy, random_weight_xy]).T

    # Main source setup
    for i, x_src in enumerate(x_positions):
        center_time = pre_delay + i * delay_between_sources
        center_sample = int(round(center_time * sampling_rate))

        start_idx = center_sample - half_samples
        end_idx   = center_sample + half_samples

        # Initialize empty arrays for the delayed wavelets
        ricker_delayed = np.zeros(len(t_sim))
        # morlet_delayed = np.zeros(len(t_sim))
        # gaussian_delayed = np.zeros(len(t_sim))

        if end_idx > 0 and start_idx < len(t_sim):
            sim_start = max(0, start_idx)
            sim_end   = min(len(t_sim), end_idx)
            wav_start = max(0, -start_idx)
            wav_end   = wav_start + (sim_end - sim_start)
            
            # Shift all three wavelets into the simulation time window
            ricker_delayed[sim_start:sim_end]   = ricker_base[wav_start:wav_end]


        # Sanity check for clipping (we only need to check one since they share dimensions)
        n_nonzero = np.count_nonzero(ricker_delayed)
        expected  = len(ricker_base)
        if n_nonzero < expected - 2:
            print(f"  WARNING source {i}: only {n_nonzero}/{expected} samples written - wavelets clipped!")

        # Construct stf array
        stf_vector_array = np.array([
            ricker_delayed   * (base_mxx * weight_array[i, 0]),
            ricker_delayed   * (base_myy * weight_array[i, 1]),
            ricker_delayed   * (base_mxy * weight_array[i, 2])
        ]).T

        # stf setup
        stf = sc.stf.Custom.from_array(
            array=stf_vector_array,
            sampling_rate_in_hertz=sampling_rate,
            start_time_in_seconds=0.0, 
        )

        # plotting_steps = np.arange(0, len(x_positions), 5)
        # if i in plotting_steps:
        #     print(f"Source {i} at x={x_src:.1f} m | delay={center_time:.4f} s | "
        #           f"weight xx:{weight_array[i,0]:.4f} yy:{weight_array[i,1]:.4f} xy:{weight_array[i,2]:.4f}")
            
        
            # stf.plot()
            # display(plt.gcf())
            # plt.close()

        src = sc.source.cartesian.MomentTensorPoint2D(
            x=x_src,
            y=y_src,
            mxx=1.0,
            myy=1.0,
            mxy=1.0,
            source_time_function=stf,
        )
        srcs.append(src)
        
    print(f"Generated {len(srcs)} sources for vprop = {target_vprop}")
    return srcs

**Salvus Execution functions**

In [ ]:
def run_simulation(p, sim_name, sources, end_time):
    """Sets up the waveform configuration and runs Salvus Flow."""
    # Define receivers, boundary conditions, wavefield outputs
    
        # Initialize simulation object, pass multiple sources
    sim = sc.simulation.Waveform(mesh=mesh_3layer, sources=srcs)

    # physics parameters 
    sim.physics.wave_equation.end_time_in_seconds = 6.545
    sim.physics.wave_equation.start_time_in_seconds = -0.2

    abc_sides = sc.boundary.Absorbing(
        width_in_meters=float(abc_thickness), 
        side_sets=["x0", "x1"],
        taper_amplitude=max_frequency * 2.0,
        side_sets_are_axis_aligned=True
    )

    abc_bottom = sc.boundary.Absorbing(
        width_in_meters=1.0, 
        side_sets=["y0"],
        taper_amplitude=max_frequency * 2.0,
        side_sets_are_axis_aligned=True
    )

    sim.physics.wave_equation.boundaries = [abc_sides, abc_bottom]

    sim.output.volume_data.format = "hdf5"
    sim.output.volume_data.fields = ["displacement", "velocity"]
    sim.output.volume_data.filename = "volume_data_output.h5"
    sim.output.volume_data.sampling_interval_in_time_steps = 50


    moving_source_output_folder = str(pathlib.Path(PROJECT_DIR) / "custom_job_moving_source_all_sources")

    print(f"Launching simulation. Outputs will be copied to: {moving_source_output_folder}")

        
        p.add_to_project(sn.EventConfiguration(waveform_simulation=sim), event_name=sim_name)
        p.simulations.launch(event_name=sim_name, site_name=os.environ.get("salome_remote"))
        p.simulations.query(block=True)
        
        # Return the path to the wavefield output for plotting
        h5_path = p.simulations.get_simulation_output_directory(sim_name) / "volume_data_output.h5"
        return h5_path